In [106]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
import optuna
import sys
from pathlib import Path
import torch # Torch needs to be installed manually if you want cuda. See installation command on pytorch website

DIR_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(DIR_ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [107]:
SEED = 987
TRACK = "off_market"                     # "off_market" or "list_aware"
VAL_START = pd.Timestamp("2025-06-01")    # sales on or after this date are validation
N_TRIALS = 10                            # optuna trials
N_ESTIMATORS = 2000                      # max boosting rounds per trial
EARLY_STOPPING_ROUNDS = 100
K_FOLDS = 5                              # time series cross validation folds
DEBUG = True                             # print MdAPE and MAPE for every fold

In [108]:
import data

listings = data.load("listings")
off_market = data.load("off_market")
list_aware = data.load("list_aware")

In [109]:
PROPERTY_TYPES = ["Condo Apt", "Detached", "Semi-Detached", "Condo Townhouse",
                  "Freehold Townhouse", "Co-Op Apt"]


def square_feet(house_area):
    """house_area is text, either one number ("622") or a range ("900-1099").
    Return a number: the number itself, or the middle of the range."""
    parts = house_area.astype(str).str.split("-", expand=True).reindex(columns=[0, 1])
    low = pd.to_numeric(parts[0], errors="coerce")
    high = pd.to_numeric(parts[1], errors="coerce").fillna(low)
    return (low + high) / 2

def make_season(date):
    """Return a season for a given date."""
    month = date.dt.month
    return pd.cut(month, bins=[0, 3, 6, 9, 12], labels=["Winter", "Spring", "Summer", "Fall"], right=True)

def off_market_features(homes):
    """Features for homes with no list price."""
    X = pd.DataFrame(index=homes.index)
    X["square_feet"] = square_feet(homes["house_area"])
    X["bedrooms"] = homes["bedrooms"]
    X["washrooms"] = homes["washrooms"]
    X["parking"] = homes["parking_total"]
    X["lot_front"] = homes["lot_front"]
    X["lot_depth"] = homes["lot_depth"]
    X["latitude"] = homes["latitude"]
    X["longitude"] = homes["longitude"]
    # Months since January 2021, so the model can follow prices over time.
    X["months"] = (homes["date_listed"] - pd.Timestamp("2021-01-01")).dt.days / 30.44
    X["community"] = homes["community"].astype("category")
    X["house_type"] = homes["house_type_name"].astype("category")
    X["house_style"] = homes["house_style"].astype("category")
    X["season"] = make_season(homes["date_listed"])
    return X


def list_aware_features(homes):
    """Features for homes that have a list price: everything above, plus the list price."""
    X = off_market_features(homes)
    X["list_price"] = homes["list_price"]
    return X

In [110]:
make_features = off_market_features if TRACK == "off_market" else list_aware_features

train = listings[listings["date_sold"] < VAL_START]
val = listings[listings["date_sold"] >= VAL_START]

X_train = make_features(train)
X_val = make_features(val)

y_train = np.log(train["sold_price"])
y_val = np.log(val["sold_price"])

print(f"train: {len(train):,} sales up to {train['date_sold'].max().date()}")
print(f"val:   {len(val):,} sales from {val['date_sold'].min().date()}")

train: 127,762 sales up to 2025-05-31
val:   22,238 sales from 2025-06-01


In [111]:
# TODO: Try different losses
def mdape(y_log, pred_log):
    return 100 * np.median(np.abs(np.exp(pred_log) / np.exp(y_log) - 1))

loss = mdape

In [112]:
from sklearn.model_selection import TimeSeriesSplit


def cross_val(X, y, params, K=5, debug=False, validation_loss=mdape):
    """Walk-forward K-fold: each fold trains on every row before its validation block.
    X and y must be sorted by time (listings is sorted by date_sold).
    Returns the mean validation loss over the folds."""
    scores = [] 
    for fold, (train_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=K).split(X), start=1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        model = XGBRegressor(**params, device=DEVICE, n_estimators=N_ESTIMATORS,
                             early_stopping_rounds=EARLY_STOPPING_ROUNDS, random_state=SEED)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        pred = model.predict(X_va)
        scores.append(validation_loss(y_va, pred))

        if debug:
            error = 100 * np.abs(np.exp(pred) / np.exp(y_va) - 1)
            print(f"fold {fold}: train {len(train_idx):,}  val {len(val_idx):,}  "
                  f"MdAPE {error.median():.2f}%  MAPE {error.mean():.2f}%", flush=True)

    return np.mean(scores)

In [113]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, step=0.01),
        "max_depth": trial.suggest_int("max_depth", 3, 13),
        "subsample": trial.suggest_float("subsample", 0.1, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0, step=0.1),
        "max_bin": trial.suggest_int("max_bin", 256, 2048),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 0.1, step=0.01),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 1, 10),
        "enable_categorical": True,
    }
    score = cross_val(X_train, y_train, params, K=K_FOLDS, debug=DEBUG, validation_loss=loss)
    return score

In [114]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED), study_name="VAPE GOD VAPE GOD VAPE GOD 420 420 696969!!!!")
study.optimize(objective, n_trials=N_TRIALS)

print("best val MdAPE:", round(study.best_value, 2))
print("best params:", study.best_params)

[I 2026-10-10 19:27:33,375] A new study created in memory with name: VAPE GOD VAPE GOD VAPE GOD 420 420 696969!!!!


fold 1: train 21,297  val 21,293  MdAPE 10.29%  MAPE 12.76%
fold 2: train 42,590  val 21,293  MdAPE 10.75%  MAPE 13.26%
fold 3: train 63,883  val 21,293  MdAPE 10.05%  MAPE 12.48%
fold 4: train 85,176  val 21,293  MdAPE 10.36%  MAPE 13.27%
fold 5: train 106,469  val 21,293  MdAPE 9.97%  MAPE 12.82%


[I 2026-10-10 19:27:47,980] Trial 0 finished with value: 10.283551720082547 and parameters: {'learning_rate': 0.05, 'max_depth': 7, 'subsample': 0.9, 'colsample_bytree': 0.5, 'max_bin': 1778, 'min_child_weight': 10, 'gamma': 0.1, 'lambda': 7.103714746188992, 'alpha': 0.0013187949354214168, 'max_delta_step': 1}. Best is trial 0 with value: 10.283551720082547.


fold 1: train 21,297  val 21,293  MdAPE 10.63%  MAPE 13.07%
fold 2: train 42,590  val 21,293  MdAPE 11.27%  MAPE 13.83%
fold 3: train 63,883  val 21,293  MdAPE 10.55%  MAPE 12.96%
fold 4: train 85,176  val 21,293  MdAPE 10.63%  MAPE 13.67%
fold 5: train 106,469  val 21,293  MdAPE 10.06%  MAPE 12.78%


[I 2026-10-10 19:27:54,137] Trial 1 finished with value: 10.626686873960063 and parameters: {'learning_rate': 0.09, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'max_bin': 808, 'min_child_weight': 3, 'gamma': 0.08, 'lambda': 4.791227993546376, 'alpha': 0.0058203973119644765, 'max_delta_step': 9}. Best is trial 0 with value: 10.283551720082547.


...

fold 1: train 21,297  val 21,293  MdAPE 10.44%  MAPE 13.03%
fold 2: train 42,590  val 21,293  MdAPE 10.51%  MAPE 13.15%
fold 3: train 63,883  val 21,293  MdAPE 10.06%  MAPE 12.58%
fold 4: train 85,176  val 21,293  MdAPE 9.97%  MAPE 12.93%
fold 5: train 106,469  val 21,293  MdAPE 9.70%  MAPE 12.46%


[I 2026-10-10 19:30:07,059] Trial 8 finished with value: 10.138880842852986 and parameters: {'learning_rate': 0.02, 'max_depth': 7, 'subsample': 0.4, 'colsample_bytree': 0.8, 'max_bin': 325, 'min_child_weight': 1, 'gamma': 0.01, 'lambda': 9.311413198108736, 'alpha': 0.030559905399179667, 'max_delta_step': 8}. Best is trial 8 with value: 10.138880842852986.


fold 1: train 21,297  val 21,293  MdAPE 10.54%  MAPE 12.99%
fold 2: train 42,590  val 21,293  MdAPE 11.23%  MAPE 13.65%
fold 3: train 63,883  val 21,293  MdAPE 10.60%  MAPE 13.06%
fold 4: train 85,176  val 21,293  MdAPE 10.73%  MAPE 13.65%
fold 5: train 106,469  val 21,293  MdAPE 10.35%  MAPE 13.12%


[I 2026-10-10 19:30:10,348] Trial 9 finished with value: 10.687567656441729 and parameters: {'learning_rate': 0.09999999999999999, 'max_depth': 7, 'subsample': 1.0, 'colsample_bytree': 0.6, 'max_bin': 1331, 'min_child_weight': 4, 'gamma': 0.09, 'lambda': 2.1227964272241358, 'alpha': 0.11463214978336936, 'max_delta_step': 6}. Best is trial 8 with value: 10.138880842852986.


best val MdAPE: 10.14
best params: {'learning_rate': 0.02, 'max_depth': 7, 'subsample': 0.4, 'colsample_bytree': 0.8, 'max_bin': 325, 'min_child_weight': 1, 'gamma': 0.01, 'lambda': 9.311413198108736, 'alpha': 0.030559905399179667, 'max_delta_step': 8}


In [115]:
from optuna.importance import get_param_importances

importance = get_param_importances(study)

for param, score in importance.items():
    print(f"{param}: {score:.4f}")

max_bin: 0.3582
lambda: 0.1541
alpha: 0.1500
min_child_weight: 0.0728
learning_rate: 0.0666
gamma: 0.0533
subsample: 0.0527
max_delta_step: 0.0438
colsample_bytree: 0.0251
max_depth: 0.0234


In [116]:
model = XGBRegressor(**study.best_params,
                     enable_categorical=True,
                     device=DEVICE,
                     early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                     n_estimators=N_ESTIMATORS,
                     random_state=SEED)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

predicted = np.exp(model.predict(X_val))
error = 100 * np.abs(predicted / val["sold_price"] - 1)
print(f"{TRACK}  MdAPE {error.median():.2f}%  MAPE {error.mean():.2f}%")

off_market  MdAPE 10.18%  MAPE 13.20%


In [117]:
feature_importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
display(feature_importances)

lot_front      0.299781
washrooms      0.210975
square_feet    0.202132
community      0.079548
lot_depth      0.065690
house_type     0.048544
bedrooms       0.028991
parking        0.019616
months         0.010735
latitude       0.008895
house_style    0.008692
longitude      0.008417
season         0.007985
dtype: float32